# 工具调用 -> 模型到底看到了什么？

本笔记本把「函数调用（tool calling）」从**调用方视角**一路拉到**模型视角**，让你看清中间每一层映射：

1. 用 **OpenAI SDK** 发起一轮带工具的多轮对话（两个工具：`read_file` / `write_file`，返回**虚拟结果**）；
2. 通过传给 SDK 的 **httpx client** 把每一轮的原始请求 / 响应 JSON 抓下来，拼出这轮对话的**最终 JSON**，保存到 `conversation.json`；
3. 把这份 JSON 喂给 HuggingFace **transformers** 的 `apply_chat_template`，让它变成模型真正看到的**文本**（带 `<|im_start|>` / `<tool_call>` 等特殊标记）；
4. 再把文本变成 **token id**（一串整数）——这才是模型实际「读」到的东西。

> **结论先行**：模型从不「看见」结构化的 `tool_calls` 对象。它只看见一条扁平的 token 序列；`apply_chat_template` 就是把结构化消息压扁成这条序列的翻译器。

```
SDK / API 层 :  messages=[{role, content, tool_calls:[{function:{name, arguments}}]}, {role:"tool", content}, ...]
      │  apply_chat_template   (本模型自带的 Jinja2 模板)
      ▼
文本层       :  <|im_start|>system\n...<tools>...</tools>...<|im_end|>\n<|im_start|>assistant\n<tool_call>\n{"name":...}\n</tool_call><|im_end|>\n...
      │  tokenize (encode)
      ▼
模型层       :  [151644, 8948, 198, ..., 151657, ..., 151658, ..., 151645, ...]
```

## 0 · 全局配置（在这里填你的服务信息）

下面这几个变量是整个笔记本的入口。把你自己的 OpenAI 兼容服务地址、key、模型名填进去即可。

In [2]:
# ===== 用户配置区（在这里填你的 OpenAI 兼容服务）=====
SDK_BASE_URL = "https://api.deepseek.com/"    # OpenAI 兼容的 base url
SDK_API_KEY  = "sk--------------"  # 你的 api key
MODEL_NAME   = "deepseek-v4-flash"                   # 模型名称

# ===== 其它配置 =====
HF_TOKENIZER_PATH = "./qwen3_tokenizer"   # 本地 Qwen3 tokenizer（已提取，不含权重，离线可用）
CONVERSATION_FILE = "conversation.json"    # 最终对话 JSON 的保存路径
MAX_TURNS = 10                             # 多轮对话最多轮数（防止死循环）

## Part 1 · 用 OpenAI SDK 做带工具的多轮对话

定义两个工具：

- `read_file(path)` —— 读取文件内容（返回虚拟内容）；
- `write_file(path, content)` —— 写入文件（返回虚拟成功）。

然后让模型完成「读取 `config.json` -> 把 `timeout` 改成 30 -> 写回去」这个任务。模型会**自己决定**调用哪个工具、传什么参数。

> 没有可用的 API 服务？可以直接跳到 **Part 2**——仓库里已自带一份 `conversation.json`（由 `make_sample_conversation.py` 生成），Part 2 能独立运行。

### 1.1 工具定义 + 虚拟实现

工具的真实逻辑被替换成「虚拟实现」：`read_file` 从一个内存 dict 里取，`write_file` 写回内存 dict。重点是**结构**，不是真实 IO。

In [3]:
import json

# 工具的「虚拟文件系统」——给工具调用返回假数据
MOCK_FILES = {
    "config.json": json.dumps(
        {"timeout": 10, "retries": 3, "host": "localhost"}, ensure_ascii=False
    )
}


def read_file(path: str) -> str:
    # 读取文件内容（虚拟实现）
    content = MOCK_FILES.get(path)
    if content is None:
        return json.dumps({"error": f"file not found: {path}"}, ensure_ascii=False)
    return content


def write_file(path: str, content: str) -> str:
    # 写入文件（虚拟实现）
    MOCK_FILES[path] = content
    return json.dumps({"status": "ok", "bytes_written": len(content)}, ensure_ascii=False)


# 模型能看到的工具清单（OpenAI function-calling schema）
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取指定路径的文件内容并返回。",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "要读取的文件路径"}
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "把内容写入指定路径的文件。",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "要写入的文件路径"},
                    "content": {"type": "string", "description": "要写入的文件内容"},
                },
                "required": ["path", "content"],
            },
        },
    },
]


def dispatch_tool(name: str, arguments: dict) -> str:
    # 根据模型给的工具名 / 参数，调用对应虚拟实现，返回结果字符串
    if name == "read_file":
        return read_file(**arguments)
    if name == "write_file":
        return write_file(**arguments)
    return json.dumps({"error": f"unknown tool: {name}"}, ensure_ascii=False)

### 1.2 造一个会「偷录」原始 JSON 的 httpx client

OpenAI SDK 允许传入自定义的 `httpx.Client`。我们给它挂上**事件钩子（event hooks）**，在每次请求发出 / 响应返回时把原始 body 抓下来。这样就能看到「线上真正传的 JSON」长什么样，而不只是 SDK 解析后的对象。

In [4]:
import httpx
from openai import OpenAI

# 每一轮的原始请求体 / 响应体都存这里（顺序一一对应）
captured_requests: list[dict] = []
captured_responses: list[dict] = []


def _on_request(request: httpx.Request):
    try:
        body = json.loads(request.content) if request.content else None
    except Exception:
        body = None
    captured_requests.append({"url": str(request.url), "body": body})


def _on_response(response: httpx.Response):
    response.read()  # 确保响应体已读取，之后 .json() 走缓存
    try:
        body = response.json()
    except Exception:
        body = response.text
    captured_responses.append(body)


# 把钩子挂上，再交给 OpenAI SDK
http_client = httpx.Client(
    event_hooks={"request": [_on_request], "response": [_on_response]}
)

client = OpenAI(
    base_url=SDK_BASE_URL,
    api_key=SDK_API_KEY,
    http_client=http_client,
)

### 1.3 多轮对话循环

每一轮：

1. 把当前 `messages` + `tools` 发给模型；
2. 模型要么回复文本（结束），要么给出 `tool_calls`；
3. 若是工具调用，就**本地执行虚拟工具**，把结果以 `role="tool"` 的消息塞回去，进入下一轮。

In [5]:
def to_clean_message(msg) -> dict:
    # 把 SDK 返回的 assistant 消息整理成干净的 dict
    clean = {"role": msg.role, "content": msg.content or ""}
    if msg.tool_calls:
        clean["tool_calls"] = [
            {
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in msg.tool_calls
        ]
    return clean


messages = [
    {"role": "system", "content": "你是一个文件管理助手。"},
    {"role": "user", "content": "请读取 config.json 的内容，然后把 timeout 字段改为 30 后保存回去。"},
]

print("=== 开始多轮对话 ===\n")
for turn in range(1, MAX_TURNS + 1):
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=TOOL_SCHEMAS,
    )
    msg = resp.choices[0].message
    assistant_msg = to_clean_message(msg)
    messages.append(assistant_msg)

    if not msg.tool_calls:
        print(f"[turn {turn}] 模型给出最终回复，对话结束。")
        print("  ->", assistant_msg["content"])
        break

    # 模型要求调用工具：逐个执行（虚拟），把结果作为 tool 消息回填
    for tc in msg.tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments)
        result = dispatch_tool(name, args)
        print(f"[turn {turn}] 模型调用工具 {name}({args})")
        print(f"         虚拟返回 -> {result}")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
else:
    print("达到 MAX_TURNS 仍未结束。")

print(f"\n共抓到 {len(captured_requests)} 次请求 / {len(captured_responses)} 次响应。")

=== 开始多轮对话 ===

[turn 1] 模型调用工具 read_file({'path': 'config.json'})
         虚拟返回 -> {"timeout": 10, "retries": 3, "host": "localhost"}
[turn 2] 模型调用工具 write_file({'path': 'config.json', 'content': '{"timeout": 30, "retries": 3, "host": "localhost"}'})
         虚拟返回 -> {"status": "ok", "bytes_written": 50}
[turn 3] 模型给出最终回复，对话结束。
  -> 已完成。`config.json` 中的 `timeout` 字段已从 `10` 改为 `30`，其余字段（`retries`、`host`）保持不变，文件已保存。

共抓到 3 次请求 / 3 次响应。


### 1.4 从 httpx 抓包重建「最终对话 JSON」并保存

最后一轮请求里，`messages` 字段已经包含了之前所有轮次（系统 / 用户 / assistant 工具调用 / tool 结果）；再加上最后一轮响应里的 assistant 文本回复，就是这轮对话的**完整 JSON**。我们把它连同 `tools` 一起存到 `conversation.json`。

> 这份 JSON 是从 httpx 抓到的**线上原文**重建的，不是手写的——这正是「通过 httpx client 把对话保存下来」的含义。

In [13]:
assert captured_requests and captured_responses, "没有抓到任何请求 / 响应，Part 1 是否成功运行了？"

# 最后一轮：请求体里的 messages 是「发出去的完整上下文」
last_req_body = captured_requests[-1]["body"]      # 请求被包成 {url, body}
last_resp_body = captured_responses[-1]             # 响应直接存的是 JSON dict 本身

final_messages = list(last_req_body["messages"])
# 补上最后一轮模型回复的 assistant 消息
final_msg_raw = last_resp_body["choices"][0]["message"]
final_messages.append({
    "role": final_msg_raw["role"],
    "content": final_msg_raw.get("content") or "",
    **({"tool_calls": final_msg_raw["tool_calls"]} if final_msg_raw.get("tool_calls") else {}),
})

final_json = {
    "model": last_req_body.get("model", MODEL_NAME),
    "tools": last_req_body.get("tools", TOOL_SCHEMAS),
    "messages": final_messages,
}

with open(CONVERSATION_FILE, "w", encoding="utf-8") as f:
    json.dump(final_json, f, ensure_ascii=False, indent=2)

print(f"已保存 {CONVERSATION_FILE}，共 {len(final_messages)} 条消息。\n")
print("===== 消息角色序列 =====")
print(" -> ".join(m["role"] for m in final_messages))

print("\n===== 最后一次请求的原始 body（httpx 抓包）=====")
print(json.dumps(last_req_body, ensure_ascii=False, indent=2))

已保存 conversation.json，共 7 条消息。

===== 消息角色序列 =====
system -> user -> assistant -> tool -> assistant -> tool -> assistant

===== 最后一次请求的原始 body（httpx 抓包）=====
{
  "messages": [
    {
      "role": "system",
      "content": "你是一个文件管理助手。"
    },
    {
      "role": "user",
      "content": "请读取 config.json 的内容，然后把 timeout 字段改为 30 后保存回去。"
    },
    {
      "role": "assistant",
      "content": "",
      "tool_calls": [
        {
          "id": "call_00_5h4A8vAtZCXaIfWgazIB3645",
          "type": "function",
          "function": {
            "name": "read_file",
            "arguments": "{\"path\": \"config.json\"}"
          }
        }
      ]
    },
    {
      "role": "tool",
      "tool_call_id": "call_00_5h4A8vAtZCXaIfWgazIB3645",
      "content": "{\"timeout\": 10, \"retries\": 3, \"host\": \"localhost\"}"
    },
    {
      "role": "assistant",
      "content": "",
      "tool_calls": [
        {
          "id": "call_00_OEborHoIaqpb40GCb2eg9527",
          "type": "function",

## Part 2 · 用 `apply_chat_template` 把对话变成 token

到这一步，`conversation.json` 里还是**结构化的**：assistant 消息里嵌着 `tool_calls` 对象，tool 结果是独立消息。模型看不懂「对象」，它只看 token。

`tokenizer.apply_chat_template(messages, tools=...)` 做的事，就是按模型自带的 **chat template**（一段 Jinja2 模板）把结构化消息翻译成一条**扁平文本字符串**；再 `encode` 成 **token id 序列**。

我们用的是本地 Qwen3 tokenizer（`./qwen3_tokenizer`，不含权重，离线可用）。

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_PATH)
print("tokenizer:", type(tokenizer).__name__, "| vocab_size:", tokenizer.vocab_size)

data = json.load(open(CONVERSATION_FILE, encoding="utf-8"))
messages = data["messages"]
tools = data["tools"]
print("读入", len(messages), "条消息，", len(tools), "个工具。")

/home/leo/blog/blog/tool_call_notes/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


tokenizer: Qwen2Tokenizer | vocab_size: 151643
读入 7 条消息， 2 个工具。


### 2.1 第一层翻译：结构化消息 -> 扁平文本（`tokenize=False`）

下面是 `apply_chat_template` 渲染出的**完整文本**。注意那些尖括号标记：`<|im_start|>` / `<|im_end|>` 是「角色边界」，`<tool_call>` / `<tool_response>` 是「工具调用 / 结果」的边界。这些原本是 JSON 对象，现在全被拍平成了纯文本。

In [8]:
rendered = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False)

print(rendered)
print("\n----- 字符数:", len(rendered), "-----")

<|im_start|>system
你是一个文件管理助手。

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "read_file", "description": "读取指定路径的文件内容并返回。", "parameters": {"type": "object", "properties": {"path": {"type": "string", "description": "要读取的文件路径"}}, "required": ["path"]}}}
{"type": "function", "function": {"name": "write_file", "description": "把内容写入指定路径的文件。", "parameters": {"type": "object", "properties": {"path": {"type": "string", "description": "要写入的文件路径"}, "content": {"type": "string", "description": "要写入的文件内容"}}, "required": ["path", "content"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
请读取 config.json 的内容，然后把 timeout 字段改为 30 后保存回去。<|im_end|>
<|im_start

### 2.2 第二层翻译：文本 -> token id（模型真正「看到」的东西）

把上面的文本 `encode` 成 token id。模型实际输入就是下面这一串整数——没有「对象」，没有「字段名」，只有 ID。

In [9]:
# chat template 已经把所有特殊标记写进文本了，所以这里 add_special_tokens=False
token_ids = tokenizer.encode(rendered, add_special_tokens=False)
print("token 总数:", len(token_ids))
print("前 30 个 id:", token_ids[:30])

# 逐 token 展示（前 40 个）。Qwen 用 byte-level BPE，中文常被切成若干字节片段，属正常现象。
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("\n  id      | token")
print("  --------|--------")
for i, t in zip(token_ids[:40], tokens[:40]):
    print(f"  {i:<7} | {repr(t)}")

token 总数: 461
前 30 个 id: [151644, 8948, 198, 56568, 101909, 26898, 39352, 110498, 3407, 2, 13852, 271, 2610, 1231, 1618, 825, 476, 803, 5746, 311, 7789, 448, 279, 1196, 3239, 382, 2610, 525, 3897, 448]

  id      | token
  --------|--------
  151644  | '<|im_start|>'
  8948    | 'system'
  198     | 'Ċ'
  56568   | 'ä½ł'
  101909  | 'æĺ¯ä¸Ģä¸ª'
  26898   | 'æĸĩä»¶'
  39352   | 'ç®¡çĲĨ'
  110498  | 'åĬ©æīĭ'
  3407    | 'ãĢĤĊĊ'
  2       | '#'
  13852   | 'ĠTools'
  271     | 'ĊĊ'
  2610    | 'You'
  1231    | 'Ġmay'
  1618    | 'Ġcall'
  825     | 'Ġone'
  476     | 'Ġor'
  803     | 'Ġmore'
  5746    | 'Ġfunctions'
  311     | 'Ġto'
  7789    | 'Ġassist'
  448     | 'Ġwith'
  279     | 'Ġthe'
  1196    | 'Ġuser'
  3239    | 'Ġquery'
  382     | '.ĊĊ'
  2610    | 'You'
  525     | 'Ġare'
  3897    | 'Ġprovided'
  448     | 'Ġwith'
  729     | 'Ġfunction'
  32628   | 'Ġsignatures'
  2878    | 'Ġwithin'
  366     | 'Ġ<'
  15918   | 'tools'
  1472    | '></'
  15918   | 'tools'
  29      |

### 2.3 关键映射：结构化对象 ↔ 特殊 token

这张表是全笔记本的核心。左边是你在 SDK 里写 / 收到的**结构化对象**，右边是模型实际看到的**特殊 token**：

| 结构化输入（SDK 视角） | 渲染出的文本（模型视角） | 是否单 token | token id |
|---|---|---|---|
| 一条消息开始 | `<\|im_start\|>` | 是 | 151644 |
| 一条消息结束 | `<\|im_end\|>` | 是 | 151645 |
| `assistant.tool_calls` 里的一次调用 | `<tool_call>\n{"name":...,"arguments":...}\n</tool_call>` | 是 | `<tool_call>`=151657, `</tool_call>`=151658 |
| `{"role":"tool",...}`（工具结果） | `<\|im_start\|>user\n<tool_response>\n...\n</tool_response><\|im_end\|>` | 是 | `<tool_response>`=151665, `</tool_response>`=151666 |
| `tools=[...]`（工具清单） | 写进 system：`<tools>...json...</tools>`（纯文本） | 否（多 token） | — |
| （Qwen3 特有）思考块 | `<think>...</think>` | 是 | 151667 / 151668 |

几条要点：

- **`tool_calls` 不是字段，是一段被 `<tool_call>` 包起来的 JSON 文本。** 模型靠 `<tool_call>` 这个 token 知道「这里有个函数调用」，靠后面那段 JSON 文本读出参数。
- **`role:"tool"` 在模型眼里根本不存在**——它被改写成 `user` 角色下的 `<tool_response>` 文本块。模型只知道「用户给了我一坨工具返回」。
- **`tools` 清单**被塞进 system 消息里，作为纯文本说明（`<tools>` 不是特殊 token，是普通字符）。
- 你用 OpenAI SDK 传的结构化对象，和你训练 / 微调时喂的 token，**靠 `apply_chat_template` 这一层模板一一对应**。换了模型（Llama、Mistral…），模板不同，同样的对话会渲染成**不同的 token 序列**。

In [10]:
# 验证上面表格里的说法：这些标记确实各是 1 个 token
markers = ["<|im_start|>", "<|im_end|>", "<tool_call>", "</tool_call>",
           "<tool_response>", "</tool_response>", "<think>", "</think>"]
print(f"{'标记':20} | {'id':8} | 是否单 token")
print("-" * 45)
for m in markers:
    ids = tokenizer.encode(m, add_special_tokens=False)
    print(f"{m:20} | {str(ids):8} | {'是' if len(ids) == 1 else '否'}")

标记                   | id       | 是否单 token
---------------------------------------------
<|im_start|>         | [151644] | 是
<|im_end|>           | [151645] | 是
<tool_call>          | [151657] | 是
</tool_call>         | [151658] | 是
<tool_response>      | [151665] | 是
</tool_response>     | [151666] | 是
<think>              | [151667] | 是
</think>             | [151668] | 是


### 2.4（附加）生成最后一轮回复时，模型实际看到的输入

上面渲染的是「整段对话」。但生成**最后那条** assistant 回复时，模型看到的输入其实是**去掉最后一条 assistant** 之后、再补一个 `<|im_start|>assistant\n`（`add_generation_prompt=True`）。这才是真正的「喂进去的 prompt」——模型从这串 token 接着往后生成。

In [11]:
prompt_text = tokenizer.apply_chat_template(
    messages[:-1],            # 去掉最后一条 assistant 回复
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)
prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)

print("生成最后一轮回复时的输入 prompt：\n")
print(prompt_text)
print(f"\n该 prompt 共 {len(prompt_ids)} 个 token；整段对话共 {len(token_ids)} 个 token。")

生成最后一轮回复时的输入 prompt：

<|im_start|>system
你是一个文件管理助手。

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "read_file", "description": "读取指定路径的文件内容并返回。", "parameters": {"type": "object", "properties": {"path": {"type": "string", "description": "要读取的文件路径"}}, "required": ["path"]}}}
{"type": "function", "function": {"name": "write_file", "description": "把内容写入指定路径的文件。", "parameters": {"type": "object", "properties": {"path": {"type": "string", "description": "要写入的文件路径"}, "content": {"type": "string", "description": "要写入的文件内容"}}, "required": ["path", "content"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
请读取 config.json 的内容，然后把 timeout 字段改为 30 后保存回去

## 总结

一条「带工具的多轮对话」，在三个层面的形态：

```
SDK / API 层 :  messages=[{role, content, tool_calls:[{function:{name, arguments}}]}, {role:"tool", content}, ...]
      │  apply_chat_template   (本模型自带的 Jinja2 模板)
      ▼
文本层       :  <|im_start|>system\n...<tools>...</tools>...<|im_end|>\n<|im_start|>assistant\n<tool_call>\n{"name":...}\n</tool_call><|im_end|>\n...
      │  tokenize (encode)
      ▼
模型层       :  [151644, 8948, 198, ..., 151657, ..., 151658, ..., 151645, ...]
```

记住三件事：

1. **模型只吃 token id。** 你在 SDK 里看到的 `tool_calls`、`role`、`arguments` 都是「给人看的结构」，到了模型全是扁平的整数序列。
2. **模板决定映射。** 同样的对话，Qwen3 渲染成 `<tool_call>` 风格，换 Llama / Mistral 就是另一套标记。所谓「支持工具调用」的模型，本质是**预训练 / 微调时见过这套模板渲染出的 token 序列**。
3. **`role:"tool"` 是个 SDK 抽象。** 模型那边它被并进 `user` + `<tool_response>`，并不存在独立的「tool 角色」。

所以当你想理解「模型为什么这么调用工具 / 为什么参数错了」，最直接的办法就是把 `apply_chat_template` 的输出打印出来——那才是模型真正读到的输入。

## 附：OpenAI 官方对"工具与缓存"的要求

上面 Part 1 里，每一轮 `create()` 我们都传了**相同**的 `tools`。这不是随便写的--OpenAI 的 **Prompt Caching** 对此有明文要求。

📄 文档：<https://developers.openai.com/api/docs/guides/prompt-caching>

### 1. 缓存只认 exact prefix match，工具属于 prefix

> "Cache hits are only possible for **exact prefix matches** within a prompt. ... place static content ... at the beginning ... put variable content ... at the end. **This also applies to images and tools, which must be identical between requests.**"

`apply_chat_template` 把 `tools` 渲染进**最顶部的 system 块**，属于 prompt 的前缀。前缀缓存要求从 token 0 起逐字一致--**工具一变，前缀就变，缓存就 miss**。

### 2. 工具定义（静态）vs 工具调用历史（动态），位置不同

> "...followed by changing timestamps, **tool-call history**, or user input. If the [breakpoint] includes that changing content, the full prefix at the breakpoint differs between requests. As a result, **cached_tokens can be 0** even though the requests share thousands of identical tokens ..."

- `tools` 定义 = **静态内容**，放前面、别动；
- `tool-call history`（多轮里不断增长的调用 / 返回消息）= **动态内容**，放后面、自然增长。

这正是多轮 agent 循环能吃到前缀缓存的关键：**前面稳定、后面增长**。

### 3. 路由 hash 用前 256 token--和我们这次实测对上了

> "Requests are routed to a machine based on a hash of the initial prefix of the prompt. The hash typically uses **the first 256 tokens** ..."

我们量到这次 prompt 的 `system+tools` 块正好 **254 token**--工具就落在路由 hash 的窗口里。改工具 → 前 256 token 变 → 路由 hash 变 → **可能直接被调度到另一台机器**，连"同机缓存"的机会都没有；即便同机，前缀也不再 exact match → miss。

### 4. GPT-5.6 起的变化（补充）

> "By default, the service places an **implicit breakpoint at the latest user or tool message**. Unlike earlier models, it **does not automatically fall back to the longest matching unmarked prefix** ..."

新模型族默认断点打在"最新一条消息"，不再自动做最长公共前缀匹配。想在多轮循环里吃到缓存，需在**稳定的 system+tools 末尾**显式打 `prompt_cache_breakpoint` + 用同一个 `prompt_cache_key`，让断点后的对话历史变化不影响前面那段缓存。但前提仍是 prefix 本身稳定--**换工具照样 miss**。

### 结论

"多轮循环里保持 `tools`（和 system）稳定"不是经验之谈，而是 OpenAI 官方文档明文要求的缓存命中前提：

- 想限制某轮可用工具，优先用 `tool_choice`（服务端约束，不改 prompt，不 miss）；
- 确实需要动态工具集时，用稳定的 wrapper 工具 + 在 user 消息里以文本描述动态函数（不动 prefix），或接受 miss。